# Exercise 1: Blockchain Structure & Tamper Detection

**COMP842 Applied Blockchains and Cryptocurrencies**

- Name: Sokhour Lay
- Student ID: 25314544

## Approach

To make this exercise more unique, the blockchain is built around an academic theme rather than around generic financial transactions from the class tutorial. It is implemented as an academic records ledger, in which each block stores real academic records such as enrolments, grades, and issued credentials. The ledger uses two classes. The `Block` class stores each block's records, a SHA-256 hash, and a Merkle root over the records, while the `Blockchain` class creates the genesis block automatically, appends new blocks, validates the chain, and reports the first invalid block. The implementation is organised into six parts. The full source code is available in the GitHub repository linked on the title page.

## Library:

In [1]:
import hashlib

from datetime import datetime

### 1. SHA-256 hash helper

A helper that turns any text into a 64-character SHA-256 fingerprint. Every part below relies on it.

In [2]:
def sha256(text):
    """Return the SHA-256 hash of a string, as a 64-character hex string."""
    return hashlib.sha256(text.encode()).hexdigest()

# Quick test: same input -> same hash; one character change -> totally different
print(sha256("hello"))
print(sha256("hello!"))

2cf24dba5fb0a30e26e83b2ac5b9e29e1b161e5c1fa7425e73043362938b9824
ce06092fb948d9ffac7d1a376e404b26b7575bcc11ee05a4615fef4fec3a308b


### 2. Merkle root

Combines all of a block's transaction hashes into a single "Merkle root" hash. Changing any transaction changes this root.

In [3]:
def merkle_root(records):
    """Combine all record hashes into one 'Merkle root' hash."""
    if not records:
        return sha256("")                               # empty block -> hash of empty text

    layer = [sha256(record) for record in records]      # 1. hash every record (the leaves)

    while len(layer) > 1:                               # 2. repeat until one hash is left
        if len(layer) % 2 == 1:                         #    odd number? duplicate the last one
            layer.append(layer[-1])
        layer = [sha256(layer[i] + layer[i + 1])        # 3. hash each neighbouring pair
                 for i in range(0, len(layer), 2)]

    return layer[0]                                     # 4. the single hash left = the root

In [4]:
sample = [
    "ENROL: S2231 enrolled in English 101 (2026-S2)",
    "ENROL: S2231 enrolled in History 210 (2026-S2)",
    "GRADE: S2231, Geography 205, Final Exam, 68/100",
]
tampered = sample[:2] + ["GRADE: S2231, Geography 205, Final Exam, 95/100"]

print("Original Sample:")
print(merkle_root(sample))
print("\nTampered Sample")
print(merkle_root(tampered))

print("\nRoots match?", merkle_root(sample) == merkle_root(tampered))

Original Sample:
6740778b8b647f89e76d88b19e1bf92cd4780d652efefc2a19a9536dba72066e

Tampered Sample
67ace0c5370eaed51c293a02064cfc8365e56d414b2ade6fc693f2758eaed3bf

Roots match? False


### 3. Block class

Stores the block's transactions, the previous block's hash, its Merkle root, and its own SHA-256 hash.

In [5]:
class Block:
    def __init__(self, index, records, previous_hash):
        self.index = index
        self.records = records
        self.timestamp = datetime.now().isoformat(timespec="seconds")  # real creation time
        self.previous_hash = previous_hash
        self.merkle_root = merkle_root(records)
        self.hash = self.compute_hash()

    def compute_hash(self):
        content = f"{self.index}{self.timestamp}{self.previous_hash}{self.merkle_root}"
        return sha256(content)

#### Test Cell

In [6]:
sample_records = [
    "ENROL: S2231 enrolled in English 101 (2026-S2)",
    "GRADE: S2231, History 210, Final Exam, 88/100",
]

b = Block(1, sample_records, "0")
print("Index:      ", b.index)
print("Records:    ", b.records)
print("Merkle root:", b.merkle_root)
print("Previous hash:", b.previous_hash)
print("Block hash: ", b.hash)

Index:       1
Records:     ['ENROL: S2231 enrolled in English 101 (2026-S2)', 'GRADE: S2231, History 210, Final Exam, 88/100']
Merkle root: 4c7a012cd2d666d87e297d1995603f50c23749a3c25823876698b545a105dd19
Previous hash: 0
Block hash:  d2c5393d905a0517a40ae88c180d5a125e794493eafe822eb38a3f275681a6fa


### 4. Blockchain class

Creates the genesis block automatically, appends new blocks, validates the chain, and reports the first invalid block.

In [7]:
class Blockchain:
    def __init__(self):
        self.chain = [self._genesis()]            # start with the genesis block

    def _genesis(self):
        return Block(0, ["GENESIS: Academic Records Ledger"], "0")

    def add_block(self, records):
        previous = self.chain[-1]                 # the last block currently in the chain
        new_block = Block(len(self.chain), records, previous.hash)
        self.chain.append(new_block)

    def first_invalid_block(self):
        for i in range(len(self.chain)):          # start at 0 so the genesis block is checked too
            current = self.chain[i]
            if current.merkle_root != merkle_root(current.records):
                return i                          # a record was changed
            if current.hash != current.compute_hash():
                return i                          # block hash no longer matches its contents
            if i > 0 and current.previous_hash != self.chain[i - 1].hash:
                return i                          # link to the previous block is broken
        return None                               # no problems found

    def is_valid(self):
        return self.first_invalid_block() is None

#### Test Cell

In [8]:
ledger = Blockchain()
ledger.add_block(["ENROL: S2231 enrolled in English 101 (2026-S2)"])
ledger.add_block(["GRADE: S2231, History 210, Final Exam, 88/100"])

print("Number of blocks:", len(ledger.chain))
print("Chain valid?    :", ledger.is_valid())
print("First invalid   :", ledger.first_invalid_block())

Number of blocks: 3
Chain valid?    : True
First invalid   : None


### 5. Build a chain of 10 blocks

Create ten blocks with distinct data and confirm the whole chain is valid (this is the output *before* tampering).

In [9]:
academic_records = [
    ["ENROL: S2231 enrolled in English 101 (2026-S2)"],           # block 1
    ["ENROL: S2231 enrolled in History 210 (2026-S2)"],           # block 2
    ["GRADE: S2231, Mathematics 150, Midterm, 76/100"],           # block 3
    ["GRADE: S2231, Science 120, Lab Report, 92/100"],            # block 4
    ["GRADE: S2231, Geography 205, Final Exam, 68/100"],          # block 5
    ["GRADE: S2231, Computer Science 101, Assignment 1, 88/100"], # block 6
    ["ENROL: S2231 enrolled in Economics 110 (2026-S2)"],         # block 7
    ["COMPLETE: S2231 finished Physics 140 with grade A"],        # block 8
    ["AWARD: Certificate of Progress issued to S2231 (2026)"],    # block 9
]

ledger = Blockchain()
for records in academic_records:
    ledger.add_block(records)

#### Test Cell

In [10]:
def print_chain(blockchain):
    for block in blockchain.chain:
        print(f"Block {block.index}  |  {block.timestamp}")
        print(f"   Records     : {block.records}")
        print(f"   Merkle root : {block.merkle_root}")
        print(f"   Prev hash   : {block.previous_hash}")
        print(f"   Hash        : {block.hash}")
        print()

print_chain(ledger)
print("Total blocks :", len(ledger.chain))
print("Chain valid? :", ledger.is_valid())
print("First invalid:", ledger.first_invalid_block())

Block 0  |  2026-09-24T20:22:51
   Records     : ['GENESIS: Academic Records Ledger']
   Merkle root : e4b3bbb455e6c5967559b0367c3830a8a2d9b82ad6b9e8b39569fa2cc03eddcd
   Prev hash   : 0
   Hash        : ce8f67f5383f45a0cac84beb95034d0b19630410b1bf1738587dc3297e7e33dd

Block 1  |  2026-09-24T20:22:51
   Records     : ['ENROL: S2231 enrolled in English 101 (2026-S2)']
   Merkle root : c0e55f2bb1ac6baddf909d94b7a58f67609aaebd83e3c9f09d945a39579c6be1
   Prev hash   : ce8f67f5383f45a0cac84beb95034d0b19630410b1bf1738587dc3297e7e33dd
   Hash        : 7d9a72ab513c7c8a5c61e01b6501d900a379a4c443152f641e6570d7357da54b

Block 2  |  2026-09-24T20:22:51
   Records     : ['ENROL: S2231 enrolled in History 210 (2026-S2)']
   Merkle root : 6c338b0e58b376fc3c4b174e4bcbc1f142d5549af511cae17d20476cee0cfcce
   Prev hash   : 7d9a72ab513c7c8a5c61e01b6501d900a379a4c443152f641e6570d7357da54b
   Hash        : a4dbbb65002d6b69e6f348e6ea0612b29277c4ec5dc2cd4e13107b51f9ab9daf

Block 3  |  2026-09-24T20:22:51
   R

### 6. Tamper detection — modify block 5

Change block 5's data without recomputing its hash, then validate again. The chain should now report block 5 as the first invalid block.

In [11]:
print("=== Before tampering ===")
print("Block 5 records:", ledger.chain[5].records)
print("Chain valid?   :", ledger.is_valid())
print()

# Realistic attack: the student raises their own grade from 68 to 95
ledger.chain[5].records = ["GRADE: S2231, Geography 205, Final Exam, 95/100  <-- FORGED"]

print("=== After tampering block 5 ===")
print("Block 5 records:", ledger.chain[5].records)
print("Chain valid?   :", ledger.is_valid())
print("First invalid  :", ledger.first_invalid_block())

=== Before tampering ===
Block 5 records: ['GRADE: S2231, Geography 205, Final Exam, 68/100']
Chain valid?   : True

=== After tampering block 5 ===
Block 5 records: ['GRADE: S2231, Geography 205, Final Exam, 95/100  <-- FORGED']
Chain valid?   : False
First invalid  : 5


### Evidence to support the reflection question

#### Checking whether the following blocks are affected

In [12]:
# Check block 6 specifically, while block 5's hash and Merkle root are still the originals
b5 = ledger.chain[5]
b6 = ledger.chain[6]

print("Block 5 - tampered, hash and Merkle root NOT recalculated")
print("   records match stored Merkle root? ", b5.merkle_root == merkle_root(b5.records))
print("   stored hash matches contents?     ", b5.hash == b5.compute_hash())

print("\nBlock 6")
print("   records match stored Merkle root? ", b6.merkle_root == merkle_root(b6.records))
print("   stored hash matches contents?     ", b6.hash == b6.compute_hash())
print("   previous hash matches block 5?    ", b6.previous_hash == b5.hash)

print("\nFirst invalid block:", ledger.first_invalid_block())

Block 5 - tampered, hash and Merkle root NOT recalculated
   records match stored Merkle root?  False
   stored hash matches contents?      True

Block 6
   records match stored Merkle root?  True
   stored hash matches contents?      True
   previous hash matches block 5?     True

First invalid block: 5


In [13]:
# Now the attacker recalculates AND saves the new Merkle root and hash into block 5
b5.merkle_root = merkle_root(b5.records)   # overwrite the stored Merkle root
b5.hash = b5.compute_hash()                # overwrite the stored block hash

print("Block 5 - after recalculating AND saving the new Merkle root and hash")
print("   records match stored Merkle root? ", b5.merkle_root == merkle_root(b5.records))
print("   stored hash matches contents?     ", b5.hash == b5.compute_hash())

print("\nBlock 6")
print("   records match stored Merkle root? ", b6.merkle_root == merkle_root(b6.records))
print("   stored hash matches contents?     ", b6.hash == b6.compute_hash())
print("   previous hash matches block 5?    ", b6.previous_hash == b5.hash)

print("\nFirst invalid block:", ledger.first_invalid_block())
print("Chain valid?       :", ledger.is_valid())

Block 5 - after recalculating AND saving the new Merkle root and hash
   records match stored Merkle root?  True
   stored hash matches contents?      True

Block 6
   records match stored Merkle root?  True
   stored hash matches contents?      True
   previous hash matches block 5?     False

First invalid block: 6
Chain valid?       : False
